# Meeting Transcriber - Exploration Notebook

This notebook walks through the full pipeline: loading audio, running ASR, and analyzing the transcript.

In [ ]:
import sys
sys.path.insert(0, '..')

from datasets import load_dataset
import numpy as np
import IPython.display as ipd

## 1. Load sample audio from LibriSpeech

In [ ]:
ds = load_dataset('librispeech_asr', 'clean', split='validation[:5]', trust_remote_code=True)
sample = ds[0]
print('Columns:', ds.column_names)
print('Reference transcript:', sample['text'])

audio_array = np.array(sample['audio']['array'])
sr = sample['audio']['sampling_rate']
print(f'Duration: {len(audio_array) / sr:.2f}s  |  Sample rate: {sr}')
ipd.Audio(audio_array, rate=sr)

## 2. Transcribe with Whisper

In [ ]:
from src.asr.transcriber import MeetingTranscriber

transcriber = MeetingTranscriber(model_size='base')
result = transcriber.transcribe_from_array(audio_array, sample_rate=sr)

print('Transcription:')
print(result.full_text)
print(f'\nDetected language: {result.language}')
print(f'Segments: {len(result.segments)}')

## 3. Analyze the transcript

In [ ]:
from src.nlp.analyzer import MeetingAnalyzer

analyzer = MeetingAnalyzer()

sample_meeting = """
Good morning everyone. Let's get started with today's meeting.
We need to discuss the Q3 sales numbers and plan for Q4.
John, can you prepare the detailed report by next Monday?
Sarah will reach out to the client and schedule a demo for this week.
We should also review the current pipeline and update the CRM.
The team needs to submit the budget forecast before end of week.
Let's also plan a follow-up meeting to track these action items.
"""

action_items = analyzer.extract_action_items(sample_meeting)
print('Action items detected:')
for item in action_items:
    print(f'  - {item}')

In [ ]:
key_topics = analyzer.extract_key_topics(sample_meeting, top_n=8)
print('Key topics:', key_topics)

## 4. Full pipeline run

In [ ]:
from src.pipeline import MeetingPipeline, PipelineConfig

config = PipelineConfig(asr_model='base')
pipeline = MeetingPipeline(config)

print('Pipeline initialized.')
print('Call pipeline.run(audio_path) to process a file.')

## 5. WER evaluation (optional)

In [ ]:
from jiwer import wer

reference = sample['text'].lower()
hypothesis = result.full_text.lower()

score = wer(reference, hypothesis)
print(f'Word Error Rate: {score:.3f} ({score * 100:.1f}%)')